# Day 4 — Physics-aware adaptation and saturation-aware calibration transfer

This notebook continues Day1–Day3 without recomputing them. It uses `results/day2/feature_table_methane.csv` as input and optionally compares Day3-style mean/std alignment. The goal is to test whether simple, interpretable regime-aware or saturation-aware corrections reduce B5 transfer error, especially at high concentration.

In [1]:
from pathlib import Path
import sys

# Works when launched from project root or from notebooks/
cwd = Path.cwd().resolve()
root_candidates = [cwd, *cwd.parents, cwd.parent]
PROJECT_ROOT = None
for p in root_candidates:
    if (p / "src" / "day4_physics_aware.py").exists():
        PROJECT_ROOT = p
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find src/day4_physics_aware.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Day2 feature table exists:", (PROJECT_ROOT / "results" / "day2" / "feature_table_methane.csv").exists())

PROJECT_ROOT = C:\Users\hg\PycharmProjects\mox_calibration_transfer
Day2 feature table exists: True


## Run Day4 pipeline

The pipeline creates `results/day4/` and `figures/day4/`, checks B5 adaptation/holdout leakage for each shot scenario, and saves all required CSV/PNG/Markdown outputs.

In [2]:
from src.day4_physics_aware import run_day4, REQUIRED_FIGURES, REQUIRED_RESULTS

out = run_day4(PROJECT_ROOT)
out

C:\Users\hg\PycharmProjects\mox_calibration_transfer\src\day4_physics_aware.py:79: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["sample_id"] = np.arange(len(df))


[Feature check] selected 81 features; excluded label/metadata columns: ['board', 'conc', 'concentration', 'concentration_code', 'concentration_numeric', 'concentration_value', 'file', 'filename', 'gas', 'label', 'ppm', 'replicate', 'sample_id', 'target', 'y_numeric']
[Leakage check] shots=0, adapt=0, holdout=20, overlap=0
[Leakage check] shots=1, adapt=10, holdout=10, overlap=0
[Leakage check] shots=2, adapt=20, holdout=0, overlap=0
[Leakage check] shots=5, adapt=20, holdout=0, overlap=0
[Leakage check] shots=10, adapt=20, holdout=0, overlap=0


C:\Users\hg\PycharmProjects\mox_calibration_transfer\src\day4_physics_aware.py:298: UserWarning: No holdout samples for 2-shot; skipping evaluation.
  warnings.warn(f"No holdout samples for {shots}-shot; skipping evaluation.")
C:\Users\hg\PycharmProjects\mox_calibration_transfer\src\day4_physics_aware.py:298: UserWarning: No holdout samples for 5-shot; skipping evaluation.
  warnings.warn(f"No holdout samples for {shots}-shot; skipping evaluation.")
C:\Users\hg\PycharmProjects\mox_calibration_transfer\src\day4_physics_aware.py:298: UserWarning: No holdout samples for 10-shot; skipping evaluation.
  warnings.warn(f"No holdout samples for {shots}-shot; skipping evaluation.")


ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

## Inspect metrics

Focus on 0-shot and 1-shot-per-concentration. Higher shot settings are included only for compatibility and must not be over-interpreted if the actual B5 adaptation set saturates.

In [ ]:
import pandas as pd
metrics = pd.read_csv(PROJECT_ROOT / "results" / "day4" / "day4_metrics.csv")
metrics.sort_values(["shots_per_concentration", "rmse"]).head(30)

## Regime-specific RMSE

This table addresses whether B5 transfer error is concentration-dependent and whether high concentration remains the hard regime.

In [ ]:
regime_metrics = pd.read_csv(PROJECT_ROOT / "results" / "day4" / "day4_regime_metrics.csv")
regime_metrics.sort_values(["shots_per_concentration", "method", "regime"]).head(50)

## Feature importance stability

Stable physics-informed features across the source model and the high-concentration subset are stronger candidates for deployment than features that only work globally.

In [ ]:
fi = pd.read_csv(PROJECT_ROOT / "results" / "day4" / "day4_feature_importance.csv")
fi.sort_values("importance_mean", ascending=False).head(25)

## Debug checks

The notebook explicitly records adaptation/holdout overlap and verifies the required result and figure filenames.

In [ ]:
leak = pd.read_csv(PROJECT_ROOT / "results" / "day4" / "day4_debug_leakage_checks.csv")
print(leak)
assert (leak["overlap"] == 0).all()

missing_figs = [f for f in REQUIRED_FIGURES if not (PROJECT_ROOT / "figures" / "day4" / f).exists()]
missing_results = [f for f in REQUIRED_RESULTS if not (PROJECT_ROOT / "results" / "day4" / f).exists()]
print("missing_figs:", missing_figs)
print("missing_results:", missing_results)
assert not missing_figs and not missing_results

## Researcher observations

The generated observations file summarizes what improved, what failed, and what Day5 should investigate.

In [ ]:
print((PROJECT_ROOT / "results" / "day4" / "day4_observations.md").read_text(encoding="utf-8"))